# 02 — Profile & Validate

Re-verifies issues Q1-Q10 from `PROJECT_BRIEF.md` §4 against the real 12-month backfill (`data/raw/calls/run_ts=20260923T163957Z`, 2025-07..2026-06, 364,873 rows) rather than trusting the brief's own pre-verified figures. Every rule applied here also runs as a named, thresholded check in `pipeline/validate.py`, written to `outputs/validation_report.json` on every pipeline run.

Full detail on anything flagged here as an open question: `docs/assumptions.md` and `docs/decision_log.md`.


In [1]:
import duckdb
import json
import pandas as pd
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Auto-discover the backfill run rather than hard-coding a run_ts - anyone
# re-running `pipeline.extract --backfill` gets a fresh timestamp, and this
# notebook should still find it without editing. Picks the run_ts directory with
# the most YYYY-MM month subdirectories (the full backfill), not a one-off
# --month/--since test run or a non-timestamp run_ts like run_ts=sample_day.
import re
candidates = {
    d: len(list(d.glob("[0-9][0-9][0-9][0-9]-[0-9][0-9]")))
    for d in (REPO_ROOT / "data" / "raw" / "calls").glob("run_ts=*")
    if re.fullmatch(r"run_ts=\d{8}T\d{6}Z", d.name)
}
best_dir = max(candidates, key=candidates.get)
RUN_TS = best_dir.name.split("=", 1)[1]
print(f"using run_ts={RUN_TS} ({candidates[best_dir]} months found)")

con = duckdb.connect()
glob = str(REPO_ROOT / "data" / "raw" / "calls" / f"run_ts={RUN_TS}" / "*" / "page_*.json")
con.execute(f"CREATE OR REPLACE VIEW raw_calls AS SELECT * FROM read_json_auto('{glob}', union_by_name=true)")
con.execute("SELECT COUNT(*) AS rows FROM raw_calls").df()


using run_ts=20260923T163957Z (12 months found)


,rows
0,364873


## Q1 — Grain: unit-level, not call-level


In [2]:
grain = con.execute("""
    SELECT COUNT(*) AS unit_rows, COUNT(DISTINCT call_number) AS distinct_calls,
           ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT call_number), 3) AS rows_per_call
    FROM raw_calls
""").df()
grain


,unit_rows,distinct_calls,rows_per_call
0,364873,182495,1.999


**Confirmed.** ~2 unit rows per call, matching the brief's Q1 finding. Any KPI at call grain (e.g. M1) needs an explicit rule for which unit "counts" — handled in Phase 4's `fct_call` model, not here.


## Q2/Q3 — Priority code mix and original-vs-final changes


In [3]:
priority_domain = con.execute("""
    SELECT COALESCE(original_priority, '(blank)') AS original_priority, COUNT(*) AS n
    FROM raw_calls GROUP BY 1 ORDER BY n DESC
""").df()
priority_domain


,original_priority,n
0,3,220137
1,2,67033
2,A,28445
3,E,15557
4,1,11787
5,B,9124
6,C,7993
7,(blank),4094
8,I,654
9,T,49


Only `2` and `3` are confirmed by the dataset's own column metadata (Code 2 / Code 3). Every other value (`A,B,C,E,I,T`, and `1`) is unconfirmed — see `docs/assumptions.md` §1 for the full table and the owner this needs.


In [4]:
changed = con.execute("""
    SELECT COUNT(*) AS total,
           COUNT(*) FILTER (WHERE original_priority IS NOT NULL AND final_priority IS NOT NULL
                             AND original_priority != final_priority) AS changed,
           ROUND(100.0 * COUNT(*) FILTER (WHERE original_priority IS NOT NULL AND final_priority IS NOT NULL
                             AND original_priority != final_priority) / COUNT(*), 1) AS pct_changed
    FROM raw_calls
""").df()
changed


,total,changed,pct_changed
0,364873,91244,25.0


**~25% of rows have a different final priority than original** — matches the brief's Q3 finding closely. Confirms priority is reassessed mid-call often enough that a KPI must pick one field explicitly (`original` vs `final`) and show how the result changes — this is exactly the definition-sensitivity table Phase 4 builds.


## Q4 — Missing `on_scene_dttm`: the real breakdown, not an assumed split


In [5]:
null_onscene = con.execute("""
    SELECT COALESCE(call_final_disposition, '(null)') AS disposition, COUNT(*) AS n
    FROM raw_calls WHERE on_scene_dttm IS NULL
    GROUP BY 1 ORDER BY n DESC
""").df()
null_onscene


,disposition,n
0,Other,24362
1,Fire,24217
2,Code 2 Transport,18312
3,Cancelled,6153
4,Unable to Locate,3290
5,Code 3 Transport,2042
6,Medical Examiner,876
7,Duplicate,35
8,Multi-casualty Incident,12


The brief characterized these nulls as splitting into "cancelled / unable to locate / genuinely missing." The real breakdown shows **Cancelled + Unable to Locate are only 11.9% of the nulls** — the two largest buckets are `Other` and `Fire`, and a notable chunk (20,354 rows) are dispositioned as an actual transport despite no recorded `on_scene_dttm`. Not resolved here — flagged in `docs/assumptions.md` §2 for Phase 4 event modeling rather than silently assumed.


## Q5 — Timestamps out of order


In [6]:
pairs = [
    ("dispatch_dttm", "response_dttm"),
    ("response_dttm", "on_scene_dttm"),
    ("on_scene_dttm", "transport_dttm"),
    ("transport_dttm", "hospital_dttm"),
    ("hospital_dttm", "available_dttm"),
]
rows = []
for earlier, later in pairs:
    r = con.execute(f"""
        SELECT
            COUNT(*) FILTER (WHERE {earlier} IS NOT NULL AND {later} IS NOT NULL) AS both_present,
            COUNT(*) FILTER (WHERE {earlier} IS NOT NULL AND {later} IS NOT NULL
                             AND CAST({later} AS TIMESTAMP) < CAST({earlier} AS TIMESTAMP)) AS violations
        FROM raw_calls
    """).fetchone()
    rows.append({"pair": f"{earlier} -> {later}", "both_present": r[0], "violations": r[1],
                 "rate": round(r[1] / r[0], 6) if r[0] else None})
pd.DataFrame(rows)


,pair,both_present,violations,rate
0,dispatch_dttm -> response_dttm,355097,25,0.000070
1,response_dttm -> on_scene_dttm,285574,17,0.000060
2,on_scene_dttm -> transport_dttm,86991,14,0.000161
3,transport_dttm -> hospital_dttm,85677,687,0.008018
4,hospital_dttm -> available_dttm,85678,480,0.005602


Violation rates are all well under 1%, consistent with the brief's Q5 finding of rare but real out-of-order timestamps. `pipeline/validate.py` WARNs on `transport->hospital` and `hospital->available` (rates ~0.6-0.8%) and would FAIL if any pair exceeded 1% (`config/validation_rules.yaml`). Violating rows are excluded from interval metrics that use that pair, not dropped from the dataset.


## Q6 — Extreme outliers (p99.9)


In [7]:
outliers = con.execute("""
    SELECT
        quantile_cont(date_diff('minute', CAST(received_dttm AS TIMESTAMP), CAST(dispatch_dttm AS TIMESTAMP)), 0.999)
            AS p999_received_to_dispatch_min,
        quantile_cont(date_diff('minute', CAST(hospital_dttm AS TIMESTAMP), CAST(available_dttm AS TIMESTAMP)), 0.999)
            AS p999_hospital_to_available_min,
        MAX(date_diff('minute', CAST(received_dttm AS TIMESTAMP), CAST(dispatch_dttm AS TIMESTAMP)))
            AS max_received_to_dispatch_min,
        MAX(date_diff('minute', CAST(hospital_dttm AS TIMESTAMP), CAST(available_dttm AS TIMESTAMP)))
            AS max_hospital_to_available_min
    FROM raw_calls
    WHERE received_dttm IS NOT NULL AND dispatch_dttm IS NOT NULL
""").df()
outliers


,p999_received_to_dispatch_min,p999_hospital_to_available_min,max_received_to_dispatch_min,max_hospital_to_available_min
0,96.0,149.0,10099,775


Extreme values exist at both ends of the lifecycle, consistent with the brief's Q6 finding. These are **flagged for review, not dropped** — `config/validation_rules.yaml` lists both intervals under `outlier_p999_fields`; Phase 4's metrics use median/p90 (robust to outliers) rather than mean.


## Q7 — Freshness and late-arriving updates


This 12-month pull is a deliberate historical backfill (chosen in Phase 1 so every month has an official scorecard actual to reconcile against) — its own most recent `received_dttm` is already ~3 months old, so a wall-clock "is this stale" check isn't meaningful here by design. `pipeline/validate.py`'s freshness check recognizes this: it only compares `data_loaded_at` against wall-clock time when the pull's own data reaches near the present (i.e. `--since`/current-month pulls). A live test against a 3-day lookback pull (`--since 3`) confirmed the check correctly PASSes there. The late-arriving-update handling itself (re-pulling a trailing window, upserting by `rowid`) is a Phase 4/5 load-time concern — `extract_calls_since()` in `pipeline/extract.py` provides the pull; the upsert lands in `load.py`.


## Q8 — Third-party (PRIVATE) units and full `unit_type` mix


In [8]:
unit_types = con.execute("SELECT unit_type, COUNT(*) AS n FROM raw_calls GROUP BY 1 ORDER BY n DESC").df()
unit_types


,unit_type,n
0,ENGINE,113189
1,MEDIC,100258
2,TRUCK,38311
3,PRIVATE,31970
4,CHIEF,28244
5,CP,22147
6,RESCUE CAPTAIN,11224
7,BLS,9766
8,SUPPORT,4995
9,RESCUE SQUAD,4584


`PRIVATE` accounts for 31,970 of 364,873 rows (8.8%) over the full 12 months — confirms the brief's Q8 finding that private ambulances are a real, non-trivial share of the system. **New finding beyond the brief:** `unit_type` also includes `ENGINE`, `TRUCK`, `CHIEF`, `CP`, `RESCUE CAPTAIN`, `BLS`, `SUPPORT`, `RESCUE SQUAD` — i.e. this dataset's "first on scene" isn't necessarily an ambulance at all. See `docs/assumptions.md` §4 — left open for Phase 4 to resolve empirically against the official scorecard, the same way the clock-start question is resolved.


## Q9 — Missing `call_type_group`


In [9]:
call_type_group = con.execute("SELECT COALESCE(call_type_group,'(null)') AS g, COUNT(*) AS n FROM raw_calls GROUP BY 1 ORDER BY n DESC").df()
call_type_group


,g,n
0,Potentially Life-Threatening,179749
1,Alarm,91937
2,Non Life-threatening,70412
3,Fire,14430
4,(null),8345


8,345 nulls (2.3%) — close to the brief's Q9 finding. **New finding:** `call_type_group` also includes `Fire` (14,430 rows) and `Alarm` (91,937 rows) alongside the two EMS-relevant groups — confirming `nuek-vuh3` covers all SFFD dispatches, not EMS calls only (`docs/assumptions.md` §3). Phase 4's KPI scoping will filter explicitly to `Potentially Life-Threatening`, stated as a named rule, not an implicit `WHERE` clause.


## Q10 — Official KPI vs. raw reconstruction

Deferred to Phase 4 (`notebooks/04_metrics_reconciliation.ipynb`) — this requires the `fct_call` SQL model and the scorecard series, not just raw profiling. Not computed here to avoid a premature, unvalidated number in this notebook.


## Validation report summary


In [10]:
report = json.loads((REPO_ROOT / "outputs" / "validation_report.json").read_text())
print("overall_status:", report["overall_status"])
pd.DataFrame(report["checks"])[["rule_id", "severity", "action"]]


overall_status: WARN


,rule_id,severity,action
0,manifest_completeness,PASS,None.
1,schema_required_columns,PASS,None.
2,rowid_uniqueness,PASS,None.
3,null_rate_response_dttm,WARN,Continue; excluded from metrics that require r...
4,null_rate_on_scene_dttm,WARN,Continue; excluded from metrics that require o...
5,null_rate_call_type_group,WARN,Continue; excluded from metrics that require c...
6,timestamp_order_dispatch_dttm_before_response_...,PASS,None.
7,timestamp_order_response_dttm_before_on_scene_...,PASS,None.
8,timestamp_order_on_scene_dttm_before_transport...,PASS,None.
9,timestamp_order_transport_dttm_before_hospital...,WARN,Continue; rows where hospital_dttm < transport...


## Summary

- Structural checks (schema, rowid uniqueness, manifest completeness) all **PASS**.
- Every known data-quality issue from the brief (Q2, Q4, Q5, Q9) reproduces at a similar   order of magnitude on the real 12-month pull and is now a named, thresholded WARN rule   in `pipeline/validate.py` rather than a one-off finding.
- Two findings **beyond** the brief: the blank-priority rate is ~19x higher over 12   months than the brief's 2-month sample suggested, and `nuek-vuh3`/`unit_type` cover   more than pure ambulance EMS response (`Fire`/`Alarm` call types, non-ambulance unit   types). Both are written up in `docs/assumptions.md` as open items for Phase 4, not   silently resolved here.
- Overall validation status: **WARN** (no structural failures; six known, counted,   documented issues). Full detail: `outputs/validation_report.json`.
